# Notebook 06 — Impact business
Objectif : traduire les irritants (motifs/situations) en **priorités business**.

Ce notebook fonctionne même si tu n'as pas de churn/NPS/coûts dans le CSV :
- il calcule un **score proxy** (volume + négatif + note + tendance)
- tu peux ensuite brancher des **colonnes réelles** (coût, churn, NPS) si tu les as.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import numpy as np
import pandas as pd

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 140)

print("OK ✅")

## 1) Charger les données (robuste)
On cherche d'abord une source déjà produite par les notebooks précédents.
Priorité : `outputs/comments_clean.parquet` (le plus fiable).

In [ ]:
import glob

def _list_outputs():
    files = sorted([p.name for p in OUT_DIR.glob("*")])
    return files

def load_base_df() -> tuple[pd.DataFrame, str]:
    # 1) Parquet nettoyé (Notebook 01)
    candidates_parquet = [
        OUT_DIR / "comments_clean.parquet",
        OUT_DIR / "comments_clean.parq",
        OUT_DIR / "comments_clean.snappy.parquet",
    ]
    for p in candidates_parquet:
        if p.exists():
            return pd.read_parquet(p), str(p)

    # 2) CSV nettoyé éventuel
    candidates_csv = [
        OUT_DIR / "comments_clean.csv",
        OUT_DIR / "comments_clean_clean.csv",
    ]
    for p in candidates_csv:
        if p.exists():
            return pd.read_csv(p), str(p)

    # 3) En dernier recours : un CSV brut (dossier data ou racine)
    brute_candidates = []
    brute_candidates += [Path(p) for p in glob.glob("data/*.csv")]
    brute_candidates += [Path(p) for p in glob.glob("*.csv")]
    for p in brute_candidates:
        try:
            df = pd.read_csv(p)
            # on vérifie qu'on a au moins motif/commentaire
            if {"motif", "commentaire"}.issubset(set(df.columns)):
                return df, str(p)
        except Exception:
            pass

    raise FileNotFoundError(
        "Aucune source trouvée. Attendu : outputs/comments_clean.parquet (ou .csv) "
        "ou un CSV brut contenant au moins les colonnes 'motif' et 'commentaire'."
    )

df, used = load_base_df()
print("Source chargée:", used)
print("Nb lignes:", len(df))
print("Colonnes:", list(df.columns))
print("\nFichiers dans outputs (utile pour debug) :")
print(_list_outputs()[:40])

## 2) Normaliser / vérifier les colonnes clés

In [ ]:
REQUIRED = ["motif", "sentiment", "note"]
missing = [c for c in REQUIRED if c not in df.columns]
print("Colonnes requises manquantes:", missing)

# Nettoyage soft : motif/sentiment en minuscules, notes numériques
if "motif" in df.columns:
    df["motif"] = df["motif"].astype(str).str.strip().str.lower()

if "sentiment" in df.columns:
    df["sentiment"] = df["sentiment"].astype(str).str.strip().str.lower()

if "note" in df.columns:
    df["note"] = pd.to_numeric(df["note"], errors="coerce")

# Date -> month (utile pour trends si on veut)
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()
else:
    df["month"] = pd.NaT

df.head(3)

## 3) Construire les métriques de base par motif
- Volume
- Répartition sentiment (part négative)
- Note moyenne


In [ ]:
# Volume
vol = (
    df.groupby("motif")
      .size()
      .reset_index(name="n_comments")
      .sort_values("n_comments", ascending=False)
)
vol["share"] = vol["n_comments"] / vol["n_comments"].sum()


# Sentiment split
if "sentiment" in df.columns:
    sent = (
        df.groupby(["motif","sentiment"])
          .size()
          .reset_index(name="n")
          .pivot(index="motif", columns="sentiment", values="n")
          .fillna(0)
          .reset_index()
    )
    sent_cols = [c for c in sent.columns if c != "motif"]
    sent["total"] = sent[sent_cols].sum(axis=1)
    if "neg" in sent.columns:
        sent["neg_share"] = sent["neg"] / sent["total"].replace(0, np.nan)
    else:
        sent["neg_share"] = np.nan
else:
    sent = pd.DataFrame({"motif": vol["motif"], "neg_share": np.nan})

# Note mean
if "note" in df.columns:
    note = (
        df.groupby("motif")["note"]
          .agg(note_mean="mean", note_count="count")
          .reset_index()
    )
else:
    note = pd.DataFrame({"motif": vol["motif"], "note_mean": np.nan, "note_count": 0})

summary = (
    vol.merge(sent[["motif", "neg_share"]], on="motif", how="left")
       .merge(note, on="motif", how="left")
       .sort_values("n_comments", ascending=False)
       .reset_index(drop=True)
)

summary.head(10)

## 4) Ajouter la tendance (growth) si elle existe
Si tu as lancé le Notebook 04, tu devrais avoir :
- `outputs/block4_signals_motif.csv` (growth_ratio, delta, n_recent, n_prev)

Sinon : pas grave, on met des valeurs neutres.

In [ ]:
# Fichier produit par Notebook 04
signals_path = OUT_DIR / "block4_signals_motif.csv"
if signals_path.exists():
    signals = pd.read_csv(signals_path)
    signals["motif"] = signals["motif"].astype(str).str.lower()
    print("Signals chargés ✅", signals_path)
    summary = summary.merge(
        signals[["motif", "n_recent", "n_prev", "delta", "growth_ratio"]],
        on="motif",
        how="left"
    )
else:
    print("Signals non trouvés (normal si Notebook 04 pas encore lancé).")
    summary["n_recent"] = np.nan
    summary["n_prev"] = np.nan
    summary["delta"] = np.nan
    summary["growth_ratio"] = np.nan

summary.head(10)

## 5) Score business proxy (0–100)
On combine (pondérable) :
- **volume** (n_comments)
- **friction** (neg_share)
- **qualité** (note_penalty)
- **tendance** (growth_ratio / delta)

⚠️ Ce n'est pas une vérité absolue : c'est un **outil de tri**.


In [ ]:
def minmax(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").fillna(0.0)
    mn, mx = s.min(), s.max()
    if mn == mx:
        return pd.Series([0.0]*len(s), index=s.index)
    return (s - mn) / (mx - mn)

# Note penalty : plus la note est basse, plus l'impact est élevé
# On suppose note sur 1..5. Si ce n'est pas le cas, la normalisation absorbe.
summary["note_penalty"] = (5 - summary["note_mean"]).clip(lower=0)

# Tendance : si growth_ratio absent -> 0. Si présent -> on prend l'excès au-dessus de 1.
gr = pd.to_numeric(summary["growth_ratio"], errors="coerce")
summary["growth_excess"] = (gr - 1).clip(lower=0).fillna(0)

# Normalisation
summary["volume_norm"] = minmax(summary["n_comments"])
summary["neg_norm"] = pd.to_numeric(summary["neg_share"], errors="coerce").fillna(0.0).clip(0, 1)
summary["note_norm"] = minmax(summary["note_penalty"])
summary["growth_norm"] = minmax(summary["growth_excess"])

# Poids (tu peux ajuster)
W_VOLUME = 0.40
W_NEG    = 0.25
W_NOTE   = 0.25
W_GROWTH = 0.10

summary["business_score_0_100"] = 100 * (
    W_VOLUME*summary["volume_norm"]
    + W_NEG*summary["neg_norm"]
    + W_NOTE*summary["note_norm"]
    + W_GROWTH*summary["growth_norm"]
)

summary_sorted = summary.sort_values("business_score_0_100", ascending=False).reset_index(drop=True)
summary_sorted.head(10)[["motif","n_comments","neg_share","note_mean","growth_ratio","business_score_0_100"]]

## 6) Option : motif → situation (si disponible)
Si tu as `outputs/block2_motif_situations.csv` ou un export similaire, on peut descendre au niveau **motif+situation**.


In [ ]:
def load_motif_situations() -> tuple[pd.DataFrame|None, str|None]:
    candidates = [
        OUT_DIR / "block2_motif_situations.csv",
        OUT_DIR / "block3_motif_situations.csv",
        OUT_DIR / "motif_situations.csv",
    ]
    for p in candidates:
        if p.exists():
            df_ms = pd.read_csv(p)
            return df_ms, str(p)
    return None, None

df_ms, used_ms = load_motif_situations()

if df_ms is None:
    print("Aucun fichier motif+situation trouvé → on s'arrête au niveau motif.")
    motif_sit = None
else:
    print("Motif+situation chargé ✅", used_ms)
    # standardisation
    df_ms["motif"] = df_ms["motif"].astype(str).str.lower()
    df_ms["situation"] = df_ms["situation"].astype(str).str.lower()

    # certains fichiers ont une colonne n ou count
    if "n" not in df_ms.columns:
        if "count" in df_ms.columns:
            df_ms["n"] = df_ms["count"]
        else:
            raise ValueError("Le fichier motif_situations doit contenir une colonne 'n' ou 'count'.")

    motif_sit = df_ms.merge(
        summary_sorted[["motif","n_comments","business_score_0_100"]],
        on="motif",
        how="left"
    )
    motif_sit["share_in_motif"] = motif_sit["n"] / motif_sit["n_comments"].replace(0, np.nan)
    motif_sit["business_score_item_0_100"] = motif_sit["business_score_0_100"] * motif_sit["share_in_motif"].fillna(0)

    motif_sit = motif_sit.sort_values("business_score_item_0_100", ascending=False).reset_index(drop=True)
    motif_sit.head(15)

## 7) Exports (pour les notebooks 07/08 et Streamlit)
- `block6_business_impact_motif.csv`
- `block6_business_impact_motif_situation.csv` (si dispo)
- `block6_summary.json`


In [ ]:
summary_out = OUT_DIR / "block6_business_impact_motif.csv"
summary_sorted.to_csv(summary_out, index=False)

if motif_sit is not None:
    ms_out = OUT_DIR / "block6_business_impact_motif_situation.csv"
    motif_sit.to_csv(ms_out, index=False)
else:
    ms_out = None

summary_json = {
    "source": used,
    "n_rows": int(len(df)),
    "n_motifs": int(df["motif"].nunique()) if "motif" in df.columns else None,
    "exports": {
        "motif": str(summary_out),
        "motif_situation": str(ms_out) if ms_out else None,
    },
    "weights": {"volume": W_VOLUME, "neg": W_NEG, "note": W_NOTE, "growth": W_GROWTH},
}
with open(OUT_DIR / "block6_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, ensure_ascii=False, indent=2)

print("Exports ✅")
print("-", summary_out)
if ms_out:
    print("-", ms_out)
print("-", OUT_DIR / "block6_summary.json")